# 📈 Linear Regression — Concepts & Implementation
### Atlas Skilltech University | ML Foundations Series

---

## Aaj Hum Kya Seekhenge?

| Topic | Kya Hai? |
|-------|----------|
| **Linear Regression** | Continuous values predict karna (price, salary, score) |
| **OLS** | Best fit line mathematically dhundna |
| **Cost Function** | Model kitna galat hai, measure karna |
| **Gradient Descent** | Automatically best parameters dhundna |
| **Residuals** | Errors ko visualize karna |
| **Regularization** | Overfitting rokna (Ridge, Lasso) |
| **Cross Validation** | Model ka real-world performance check karna |

---

## Real-World Analogy 🏠

Socho tumhare paas ek **GPS tracker** hai jo road pe distance dekh ke **travel time predict** karta hai.  
Jitni zyada distance → utna zyada time. Yahi Linear Regression hai!

**Linear Regression = X ko dekh ke Y predict karo**
- X = Study Hours → Y = Exam Score  
- X = House Size → Y = House Price  
- X = Experience Years → Y = Salary

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
print('✅ Libraries load ho gayi!')

---
## 1. Introduction to Linear Regression

### Linear Regression Kya Hai?

Linear Regression ek **supervised learning algorithm** hai jo ek **straight line** fit karta hai data ke through.

**Mathematical Form:**
```
y = β₀ + β₁x₁ + β₂x₂ + ... + βₙxₙ + ε
```
- **y** = target variable (jo predict karna hai)
- **β₀** = intercept (y-axis par kahan se start karein)
- **β₁...βₙ** = coefficients (har feature ka weight)
- **x₁...xₙ** = input features
- **ε** = error term (noise)

**Simple Linear Regression** (1 feature): `y = β₀ + β₁x`  
**Multiple Linear Regression** (n features): `y = β₀ + β₁x₁ + ... + βₙxₙ`

In [ ]:
# Salary Dataset banate hain — Experience se Salary predict karna
np.random.seed(42)
experience = np.random.uniform(0, 15, 100)
salary = 25000 + 8000 * experience + np.random.normal(0, 15000, 100)
salary = np.clip(salary, 20000, None)

df_salary = pd.DataFrame({'Experience_Years': experience, 'Salary_INR': salary})
print('Salary Dataset:')
print(df_salary.head(8))
print(f'\nShape: {df_salary.shape}')
print(f'Salary Range: ₹{df_salary.Salary_INR.min():.0f} — ₹{df_salary.Salary_INR.max():.0f}')

In [ ]:
# Data Visualize karo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(df_salary.Experience_Years, df_salary.Salary_INR,
                alpha=0.7, color='steelblue', edgecolors='white', s=80)
axes[0].set_xlabel('Experience (Years)')
axes[0].set_ylabel('Salary (₹)')
axes[0].set_title('Experience vs Salary — Scatter Plot')
axes[0].grid(alpha=0.3)

# Salary distribution
axes[1].hist(df_salary.Salary_INR, bins=20, color='coral', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Salary (₹)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Salary Distribution')
axes[1].axvline(df_salary.Salary_INR.mean(), color='darkred', linestyle='--',
                label=f'Mean: ₹{df_salary.Salary_INR.mean():.0f}')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Salary Dataset — Exploratory Data Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('📊 EDA complete! Clear positive trend dikh raha hai.')

---
## 2. Ordinary Least Squares (OLS)

### OLS Kya Hai?

OLS ek **mathematical method** hai jo best fit line dhundta hai by **minimizing the sum of squared errors**.

**Analogy:** Agar tum rubber bands se data points ko line se connect karo, OLS woh line dhundta hai jahan rubber bands ki total khichaav **sabse kam** ho.

**OLS Formulas:**
```
β₁ = Σ(xᵢ - x̄)(yᵢ - ȳ) / Σ(xᵢ - x̄)²
β₀ = ȳ - β₁ * x̄
```

**Matrix Form (Multiple Regression):**
```
β = (XᵀX)⁻¹ Xᵀy
```

**Assumptions of OLS (LINE):**
- **L**inearity — X aur Y ka relationship linear ho
- **I**ndependence — observations independent hon
- **N**ormality — residuals normally distributed hon
- **E**qual variance (Homoscedasticity) — residuals ka variance constant ho

In [ ]:
# OLS Manually Implement karte hain (Scratch se)
X = df_salary.Experience_Years.values
y = df_salary.Salary_INR.values

x_mean = np.mean(X)
y_mean = np.mean(y)

# OLS formulas
numerator = np.sum((X - x_mean) * (y - y_mean))
denominator = np.sum((X - x_mean) ** 2)

beta1_manual = numerator / denominator  # slope
beta0_manual = y_mean - beta1_manual * x_mean  # intercept

print('=== OLS Results (Scratch) ===')
print(f'β₀ (Intercept): ₹{beta0_manual:,.0f}')
print(f'β₁ (Slope):     ₹{beta1_manual:,.0f} per year of experience')
print(f'\nInterpretation: Har saal experience badhne se salary ~₹{beta1_manual:,.0f} badhti hai')

# Predict karo
y_pred_manual = beta0_manual + beta1_manual * X

# Visualize
plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.6, color='steelblue', label='Actual Data', s=80)
plt.plot(X, y_pred_manual, color='red', linewidth=2.5, label=f'OLS Line: y = {beta0_manual:.0f} + {beta1_manual:.0f}x')

# Error lines (residuals)
for i in range(0, len(X), 8):
    plt.plot([X[i], X[i]], [y[i], y_pred_manual[i]], 'gray', alpha=0.5, linewidth=1)

plt.xlabel('Experience (Years)')
plt.ylabel('Salary (₹)')
plt.title('OLS Best Fit Line — Gray lines = Residuals (Errors)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---
## 3. Cost Functions

### Cost Function Kya Hai?

Cost Function batata hai ki **model kitna galat hai**. Jitna kam cost → utna better model.

| Metric | Formula | Kab Use Karo |
|--------|---------|---------------|
| **MSE** | mean((y - ŷ)²) | Outliers ko zyada penalize karna ho |
| **MAE** | mean(|y - ŷ|) | Outliers ka effect kam rakhna ho |
| **RMSE** | √MSE | Same units mein error chahiye |
| **R²** | 1 - SS_res/SS_tot | Model ka overall goodness of fit |

**R² Explanation:**
- R² = 1.0 → Perfect prediction
- R² = 0.0 → Model mean hi predict kar raha hai (useless)
- R² = 0.85 → Model 85% variance explain kar raha hai
- R² < 0 → Model mean se bhi bura hai!

In [ ]:
# Cost Functions Calculate karo
mse = mean_squared_error(y, y_pred_manual)
mae = mean_absolute_error(y, y_pred_manual)
rmse = np.sqrt(mse)
r2 = r2_score(y, y_pred_manual)

print('=== Cost Function Results ===')
print(f'MSE  (Mean Squared Error):       ₹{mse:,.0f}')
print(f'MAE  (Mean Absolute Error):      ₹{mae:,.0f}')
print(f'RMSE (Root Mean Squared Error):  ₹{rmse:,.0f}')
print(f'R²   (R-Squared):                {r2:.4f}')
print(f'\nInterpretation: Model {r2*100:.1f}% variance explain kar raha hai')
print(f'Average prediction error: ~₹{rmse:,.0f}')

In [ ]:
# MSE vs Slope — Cost Landscape visualize karo
slopes = np.linspace(2000, 14000, 200)
mse_values = []

for slope in slopes:
    intercept = y_mean - slope * x_mean
    y_pred_temp = intercept + slope * X
    mse_values.append(mean_squared_error(y, y_pred_temp))

plt.figure(figsize=(10, 5))
plt.plot(slopes, mse_values, color='purple', linewidth=2)
plt.axvline(beta1_manual, color='red', linestyle='--', linewidth=2,
            label=f'OLS Optimal Slope = {beta1_manual:.0f}')
plt.scatter([beta1_manual], [mse], color='red', s=150, zorder=5)
plt.xlabel('Slope (β₁) Value')
plt.ylabel('MSE (Cost)')
plt.title('Cost Landscape — MSE vs Slope (Minimum point = OLS Solution)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print('💡 OLS automatically is minimum point pe land karta hai!')

---
## 4. Gradient Descent

### Gradient Descent Kya Hai?

Gradient Descent ek **optimization algorithm** hai jo **iteratively** best parameters dhundta hai.

**Analogy — Mountain Climbing:**  
Imagine tum ek foggy mountain par ho. Tum neeche jaana chahte ho (minimum cost).  
Gradient Descent bar bar poochta hai: **"Mera current position pe slope kahan neeche hai?"**  
Phir uss direction mein ek chhota step leta hai.

**Update Rule:**
```
β₁ = β₁ - α * ∂MSE/∂β₁
β₀ = β₀ - α * ∂MSE/∂β₀
```
- **α (Learning Rate):** Step size — kitna bada step lena hai
  - Too large → overshoot (miss the minimum)
  - Too small → bahut slow convergence

**Gradients:**
```
∂MSE/∂β₁ = -(2/n) * Σ xᵢ(yᵢ - ŷᵢ)
∂MSE/∂β₀ = -(2/n) * Σ (yᵢ - ŷᵢ)
```

In [ ]:
# Gradient Descent Scratch se implement karo
# Data normalize karo first (GD ke liye important!)
X_norm = (X - X.mean()) / X.std()
y_norm = (y - y.mean()) / y.std()
n = len(X_norm)

# Initialize parameters
beta0_gd = 0.0
beta1_gd = 0.0
learning_rate = 0.01
epochs = 1000
cost_history = []

for epoch in range(epochs):
    # Predictions
    y_pred_gd = beta0_gd + beta1_gd * X_norm
    
    # Compute cost (MSE)
    cost = np.mean((y_norm - y_pred_gd) ** 2)
    cost_history.append(cost)
    
    # Compute gradients
    d_beta0 = -(2/n) * np.sum(y_norm - y_pred_gd)
    d_beta1 = -(2/n) * np.sum(X_norm * (y_norm - y_pred_gd))
    
    # Update parameters
    beta0_gd -= learning_rate * d_beta0
    beta1_gd -= learning_rate * d_beta1

print(f'=== Gradient Descent Results ===')
print(f'Final β₀: {beta0_gd:.6f} (normalized space)')
print(f'Final β₁: {beta1_gd:.6f} (normalized space)')
print(f'Final MSE: {cost_history[-1]:.6f}')
print(f'OLS se milta hai? Yes! GD bhi same solution dhundta hai.')

# Convergence plot
plt.figure(figsize=(10, 5))
plt.plot(range(epochs), cost_history, color='blue', linewidth=2)
plt.xlabel('Epoch (Iteration)')
plt.ylabel('MSE Cost')
plt.title('Gradient Descent — Cost Convergence')
plt.grid(alpha=0.3)
plt.annotate(f'Final Cost: {cost_history[-1]:.4f}',
             xy=(epochs*0.8, cost_history[-1]*1.5),
             fontsize=11, color='red')
plt.show()

In [ ]:
# Learning Rate ka Effect
learning_rates = [0.001, 0.01, 0.1, 0.5]
colors = ['blue', 'green', 'orange', 'red']

plt.figure(figsize=(12, 6))
for lr, color in zip(learning_rates, colors):
    b0, b1 = 0.0, 0.0
    costs = []
    for _ in range(200):
        y_p = b0 + b1 * X_norm
        c = np.mean((y_norm - y_p) ** 2)
        costs.append(c)
        d0 = -(2/n) * np.sum(y_norm - y_p)
        d1 = -(2/n) * np.sum(X_norm * (y_norm - y_p))
        b0 -= lr * d0
        b1 -= lr * d1
        if c > 1e6:  # diverging
            break
    plt.plot(costs[:100], color=color, label=f'LR = {lr}', linewidth=2)

plt.xlabel('Epoch')
plt.ylabel('MSE Cost')
plt.title('Learning Rate ka Effect on Convergence')
plt.legend()
plt.ylim(0, 2)
plt.grid(alpha=0.3)
plt.show()
print('💡 LR=0.001 too slow, LR=0.5 oscillates/diverges, LR=0.01-0.1 sweet spot!')

---
## 5. Implementation with Scikit-Learn

### Sklearn ka LinearRegression — Simple aur Multiple

Ab hum OLS scratch se banana band karte hain — sklearn sab kuch automatically karta hai!

In [ ]:
# Multiple Feature Dataset — House Price Prediction
np.random.seed(42)
n_samples = 200

size_sqft = np.random.uniform(500, 3000, n_samples)
bedrooms = np.random.choice([1, 2, 3, 4, 5], n_samples)
age_years = np.random.uniform(0, 40, n_samples)
distance_km = np.random.uniform(1, 30, n_samples)

price = (120 * size_sqft + 500000 * bedrooms - 20000 * age_years
         - 150000 * distance_km + np.random.normal(0, 200000, n_samples))
price = np.clip(price, 500000, None)

df_house = pd.DataFrame({
    'Size_sqft': size_sqft,
    'Bedrooms': bedrooms,
    'Age_years': age_years,
    'Distance_km': distance_km,
    'Price': price
})

print('House Price Dataset:')
print(df_house.head())
print(f'\nShape: {df_house.shape}')
print(f'Price Range: ₹{price.min():,.0f} — ₹{price.max():,.0f}')

In [ ]:
# Train-Test Split + sklearn LinearRegression
X_house = df_house.drop('Price', axis=1)
y_house = df_house['Price']

X_train, X_test, y_train, y_test = train_test_split(
    X_house, y_house, test_size=0.2, random_state=42
)

# Model train karo
model = LinearRegression()
model.fit(X_train, y_train)

# Predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

print('=== sklearn LinearRegression ===')
print(f'Intercept (β₀): ₹{model.intercept_:,.0f}')
print('\nCoefficients:')
for feat, coef in zip(X_house.columns, model.coef_):
    print(f'  {feat:15s}: ₹{coef:,.0f}')

print(f'\nTrain R²:  {r2_score(y_train, y_pred_train):.4f}')
print(f'Test R²:   {r2_score(y_test, y_pred_test):.4f}')
print(f'Test RMSE: ₹{np.sqrt(mean_squared_error(y_test, y_pred_test)):,.0f}')

---
## 6. Residual Plots

### Residuals Kya Hote Hain?

**Residual = Actual Value − Predicted Value**  
`eᵢ = yᵢ − ŷᵢ`

**Good Model ke Residuals:**
- Random scatter hona chahiye (no pattern)
- Mean ≈ 0 hona chahiye
- Normally distributed hona chahiye (histogram bell curve)

**Bad Patterns:**
- Funnel shape → Heteroscedasticity (unequal variance)
- Curve → Non-linearity (linear model fit nahi kiya)
- Clusters → Missing categorical feature

In [ ]:
residuals = y_test.values - y_pred_test

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Residuals vs Predicted (Method 1 - Manual)
axes[0].scatter(y_pred_test, residuals, alpha=0.6, color='steelblue', s=60)
axes[0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Predicted Price (₹)')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted\n(Method 1: Manual Scatter)')
axes[0].grid(alpha=0.3)

# Plot 2: Residual Histogram
axes[1].hist(residuals, bins=25, color='coral', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Residual Value')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution\n(Should be Normal/Bell-shaped)')
axes[1].grid(alpha=0.3)

# Plot 3: Actual vs Predicted
axes[2].scatter(y_test, y_pred_test, alpha=0.6, color='purple', s=60)
min_val = min(y_test.min(), y_pred_test.min())
max_val = max(y_test.max(), y_pred_test.max())
axes[2].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2,
             label='Perfect Prediction Line')
axes[2].set_xlabel('Actual Price (₹)')
axes[2].set_ylabel('Predicted Price (₹)')
axes[2].set_title('Actual vs Predicted')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle('Residual Analysis — Method 1 (Manual)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Method 2 — Modern sklearn way + seaborn residplot
from sklearn.metrics import PredictionErrorDisplay

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PredictionErrorDisplay (new sklearn way)
PredictionErrorDisplay.from_predictions(
    y_test, y_pred_test,
    kind='actual_vs_predicted',
    ax=axes[0]
)
axes[0].set_title('Actual vs Predicted\n(Method 2: PredictionErrorDisplay)')

# seaborn residplot
sns.residplot(
    x=y_pred_test, y=y_test.values,
    lowess=True,
    scatter_kws={'alpha': 0.5},
    line_kws={'color': 'red'},
    ax=axes[1]
)
axes[1].set_xlabel('Predicted Price (₹)')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot with LOWESS\n(Method 2: seaborn residplot)')
axes[1].grid(alpha=0.3)

plt.suptitle('Residual Analysis — Method 2 (sklearn + seaborn)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('💡 Red LOWESS line ≈ flat hai → Good! Koi non-linear pattern nahi.')

---
## 7. Coefficient Interpretation

### Coefficients Ko Kaise Padhein?

**Coefficient (β) ka matlab:**
> "Agar yeh feature 1 unit badhe, aur baaki sab features constant rahein, toh Y kitna badlega?"

**Important:** Direct comparison ke liye features ko **standardize** karna padta hai!

In [ ]:
# Standardized Coefficients — real importance compare karo
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_scaled = LinearRegression()
model_scaled.fit(X_train_scaled, y_train)

coef_df = pd.DataFrame({
    'Feature': X_house.columns,
    'Raw_Coef': model.coef_,
    'Std_Coef': model_scaled.coef_
}).sort_values('Std_Coef', key=abs, ascending=False)

print('Coefficient Comparison:')
print(coef_df.to_string(index=False))
print('\nInterpretation:')
for _, row in coef_df.iterrows():
    direction = '📈 increases' if row.Std_Coef > 0 else '📉 decreases'
    print(f'  {row.Feature}: price {direction} by ₹{abs(row.Raw_Coef):,.0f} per unit')

# Visualization
plt.figure(figsize=(10, 5))
colors = ['green' if c > 0 else 'red' for c in coef_df.Std_Coef]
plt.barh(coef_df.Feature, coef_df.Std_Coef, color=colors, alpha=0.8, edgecolor='black')
plt.axvline(0, color='black', linewidth=1)
plt.xlabel('Standardized Coefficient')
plt.title('Feature Importance via Standardized Coefficients\nGreen = Positive Effect, Red = Negative Effect')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8. Bias-Variance Tradeoff

### Bias vs Variance Kya Hai?

**Analogy — Archery:**
- **High Bias:** Consistently miss karo — ek jagah pe cluster, but wrong target
- **High Variance:** Kabhi yahan, kabhi wahan — unpredictable, scattered
- **Sweet Spot:** Low bias + Low variance = Bullseye!

| | Low Variance | High Variance |
|--|--|--|
| **Low Bias** | ✅ Ideal Model | Overfitting (complex model) |
| **High Bias** | Underfitting (simple model) | ❌ Worst case |

**Linear Regression aur Polynomial Regression:**
- Degree 1 (line) = High Bias, Low Variance
- Degree 10 (wavy) = Low Bias, High Variance

In [ ]:
# Polynomial Regression se Bias-Variance Tradeoff demonstrate karo
np.random.seed(42)
X_poly = np.sort(np.random.uniform(0, 10, 50))
y_poly = 2 * X_poly - 0.3 * X_poly**2 + np.random.normal(0, 3, 50)

X_poly_2d = X_poly.reshape(-1, 1)
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_poly_2d, y_poly, test_size=0.3, random_state=42
)

degrees = [1, 2, 5, 10]
train_errors, test_errors = [], []

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
X_line = np.linspace(0, 10, 300).reshape(-1, 1)

for i, deg in enumerate(degrees):
    pipe = Pipeline([('poly', PolynomialFeatures(deg)), ('lr', LinearRegression())])
    pipe.fit(X_train_p, y_train_p)
    
    train_r2 = r2_score(y_train_p, pipe.predict(X_train_p))
    test_r2 = r2_score(y_test_p, pipe.predict(X_test_p))
    train_errors.append(train_r2)
    test_errors.append(test_r2)
    
    axes[i].scatter(X_poly, y_poly, alpha=0.6, color='steelblue', s=40)
    axes[i].plot(X_line, pipe.predict(X_line), 'r-', linewidth=2)
    axes[i].set_title(f'Degree {deg}\nTrain R²={train_r2:.2f}, Test R²={test_r2:.2f}')
    axes[i].set_ylim(-10, 20)
    axes[i].grid(alpha=0.3)
    
    if deg == 1:
        axes[i].set_xlabel('⬆ High Bias (Underfitting)')
    elif deg == 2:
        axes[i].set_xlabel('✅ Sweet Spot')
    elif deg == 10:
        axes[i].set_xlabel('⬆ High Variance (Overfitting)')

plt.suptitle('Bias-Variance Tradeoff — Polynomial Regression', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Regularization Overview

### Regularization Kyun Chahiye?

Jab model **overfit** karta hai (training pe perfect, testing pe bura), toh regularization use karte hain.

**Regularization = Extra Penalty Add Karo Large Coefficients Par**

| Method | Formula | Effect | Kab Use |
|--------|---------|--------|----------|
| **Ridge (L2)** | MSE + α * Σβ² | Coefficients small karo, koi zero nahi | Multicollinearity |
| **Lasso (L1)** | MSE + α * Σ|β| | Coefficients zero kar sakta hai (feature selection) | Many irrelevant features |
| **ElasticNet** | MSE + α₁Σ|β| + α₂Σβ² | Ridge + Lasso both | General purpose |

**Alpha (α) Parameter:**
- α = 0 → Regular Linear Regression
- α → ∞ → All coefficients → 0 (complete underfitting)

In [ ]:
# Ridge vs Lasso vs ElasticNet comparison
alphas = [0.01, 0.1, 1, 10, 100, 1000]

ridge_r2, lasso_r2, en_r2 = [], [], []

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    lasso = Lasso(alpha=alpha)
    en = ElasticNet(alpha=alpha, l1_ratio=0.5)
    
    for m, r in [(ridge, ridge_r2), (lasso, lasso_r2), (en, en_r2)]:
        m.fit(X_train_scaled, y_train)
        r.append(r2_score(y_test, m.predict(X_test_scaled)))

plt.figure(figsize=(10, 6))
plt.semilogx(alphas, ridge_r2, 'b-o', label='Ridge (L2)', linewidth=2)
plt.semilogx(alphas, lasso_r2, 'r-s', label='Lasso (L1)', linewidth=2)
plt.semilogx(alphas, en_r2, 'g-^', label='ElasticNet', linewidth=2)

lr_r2 = r2_score(y_test, model_scaled.predict(X_test_scaled))
plt.axhline(lr_r2, color='black', linestyle='--', label=f'Linear Reg (no reg): {lr_r2:.3f}')

plt.xlabel('Alpha (Regularization Strength) — Log Scale')
plt.ylabel('Test R²')
plt.title('Regularization Effect — Ridge vs Lasso vs ElasticNet')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Lasso Feature Selection demonstrate karo — many features dataset
np.random.seed(42)
n, p = 100, 15  # 100 samples, 15 features
X_many = np.random.randn(n, p)
# Only features 0, 2, 5 matter
y_many = 3 * X_many[:, 0] + 1.5 * X_many[:, 2] - 2 * X_many[:, 5] + np.random.randn(n)

ridge_many = Ridge(alpha=0.1)
lasso_many = Lasso(alpha=0.1)
ridge_many.fit(X_many, y_many)
lasso_many.fit(X_many, y_many)

feat_names = [f'F{i}' for i in range(p)]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model_r, name, color in [
    (axes[0], ridge_many, 'Ridge (L2)', 'steelblue'),
    (axes[1], lasso_many, 'Lasso (L1)', 'coral')
]:
    ax.bar(feat_names, model_r.coef_, color=color, edgecolor='black', alpha=0.8)
    ax.axhline(0, color='black', linewidth=1)
    ax.set_title(f'{name} Coefficients')
    ax.set_xlabel('Feature')
    ax.set_ylabel('Coefficient Value')
    ax.grid(alpha=0.3)

n_zero = np.sum(lasso_many.coef_ == 0)
plt.suptitle(f'Ridge vs Lasso: Lasso ne {n_zero}/{p} features ko ZERO kar diya (Feature Selection!)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. Feature Scaling

### Feature Scaling Kyun Zaroori Hai?

**Problem:** Agar ek feature 0-1 range mein hai aur doosra 100,000-500,000 range mein,  
toh model zyada range wale feature ko zyada important maanta hai.

**Linear Regression aur Scaling:**
- OLS coefficients scaling se affect hote hain (interpretation ke liye)
- Gradient Descent ke liye scaling bahut zaroori hai (convergence speed)
- Regularization (Ridge/Lasso) ke liye scaling critical hai

| Scaler | Formula | Kab Use |
|--------|---------|----------|
| **StandardScaler** | (x - mean) / std | Normal distribution data |
| **MinMaxScaler** | (x - min) / (max - min) | Bounded range chahiye (0 to 1) |
| **RobustScaler** | (x - median) / IQR | Outliers present hain |

In [ ]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler

# Without scaling vs With scaling — coefficient comparison
models = {
    'No Scaling': LinearRegression(),
    'StandardScaler': Pipeline([('sc', StandardScaler()), ('lr', LinearRegression())]),
    'MinMaxScaler': Pipeline([('sc', MinMaxScaler()), ('lr', LinearRegression())]),
}

print('=== R² Comparison (Scaling Effect) ===')
for name, mdl in models.items():
    mdl.fit(X_train, y_train)
    r2_tr = r2_score(y_train, mdl.predict(X_train))
    r2_te = r2_score(y_test, mdl.predict(X_test))
    print(f'{name:20s}: Train R²={r2_tr:.4f}, Test R²={r2_te:.4f}')

print('\n💡 R² same hai! Scaling R² nahi badlata, sirf coefficients aur GD speed badlata hai.')
print('💡 But Ridge/Lasso ke liye scaling MUST hai otherwise fairness nahi!')

---
## 11. Cross Validation

### Cross Validation Kya Hai?

**Problem:** Agar hum ek baar split karein, shayad unlucky split mile!  
**Solution:** Data ko multiple times split karo aur average score lo.

**K-Fold Cross Validation:**
- Data ko K equal parts mein split karo
- Har iteration mein 1 part test, baaki K-1 parts train
- K scores ka average lo

**Types:**
- **K-Fold**: Standard, good for large datasets
- **Stratified K-Fold**: Class balance maintain kare (classification ke liye)
- **Leave-One-Out (LOO)**: K = n, very thorough but slow

In [ ]:
# Cross Validation implement karo
X_all = df_house.drop('Price', axis=1)
y_all = df_house['Price']

pipeline_cv = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LinearRegression())
])

# 5-Fold CV
cv_scores = cross_val_score(pipeline_cv, X_all, y_all, cv=5, scoring='r2')
cv_rmse = cross_val_score(pipeline_cv, X_all, y_all, cv=5,
                           scoring='neg_root_mean_squared_error')

print('=== 5-Fold Cross Validation Results ===')
print(f'R² Scores per fold: {cv_scores.round(4)}')
print(f'Mean R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'\nRMSE per fold: {(-cv_rmse).round(0)}')
print(f'Mean RMSE: ₹{(-cv_rmse).mean():,.0f} ± ₹{(-cv_rmse).std():,.0f}')

# Visualize fold scores
plt.figure(figsize=(10, 5))
plt.bar(range(1, 6), cv_scores, color='steelblue', edgecolor='black', alpha=0.8)
plt.axhline(cv_scores.mean(), color='red', linestyle='--', linewidth=2,
            label=f'Mean R² = {cv_scores.mean():.4f}')
plt.fill_between(range(1, 6),
                 cv_scores.mean() - cv_scores.std(),
                 cv_scores.mean() + cv_scores.std(),
                 alpha=0.2, color='red', label='±1 Std Dev')
plt.xlabel('Fold Number')
plt.ylabel('R² Score')
plt.title('5-Fold Cross Validation — R² per Fold')
plt.legend()
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# CV for Model Selection — LR vs Ridge vs Lasso
models_cv = {
    'Linear Regression': Pipeline([('sc', StandardScaler()), ('lr', LinearRegression())]),
    'Ridge (α=1)': Pipeline([('sc', StandardScaler()), ('lr', Ridge(alpha=1))]),
    'Lasso (α=1)': Pipeline([('sc', StandardScaler()), ('lr', Lasso(alpha=1))]),
    'ElasticNet': Pipeline([('sc', StandardScaler()), ('lr', ElasticNet(alpha=1, l1_ratio=0.5))]),
}

print('=== Model Comparison via 5-Fold CV ===')
comparison = []
for name, mdl in models_cv.items():
    scores = cross_val_score(mdl, X_all, y_all, cv=5, scoring='r2')
    comparison.append({'Model': name, 'Mean R²': scores.mean(), 'Std': scores.std()})

comp_df = pd.DataFrame(comparison).sort_values('Mean R²', ascending=False)
print(comp_df.to_string(index=False))

---
## Summary — Aaj Kya Seekha?

| Concept | Key Takeaway |
|---------|-------------|
| **Linear Regression** | y = β₀ + β₁x₁ + ... straight line fit |
| **OLS** | Mathematically best line = sum of squared errors minimize karo |
| **Cost Functions** | MSE (penalizes outliers), MAE (robust), R² (goodness of fit) |
| **Gradient Descent** | Iteratively best parameters dhundta hai, learning rate critical |
| **sklearn** | `LinearRegression().fit(X, y)` — sab kuch automatic |
| **Residuals** | Random scatter = good; pattern = problem |
| **Coefficients** | β = 1 unit change mein Y ka expected change |
| **Bias-Variance** | Simple model = high bias; Complex model = high variance |
| **Ridge/Lasso** | Regularization = overfitting rokna; Lasso = feature selection bhi |
| **Feature Scaling** | GD aur Regularization ke liye critical |
| **Cross Validation** | Realistic performance estimate, overfitting ka pata lagao |

## Real-World Applications

| Industry | Use Case |
|----------|----------|
| **Finance** | Stock price prediction, loan amount estimation |
| **Real Estate** | House price prediction (Magicbricks, 99acres use it) |
| **HR/Recruiting** | Salary benchmarking based on experience |
| **Healthcare** | Blood pressure from age/BMI, drug dosage |
| **E-commerce** | Demand forecasting, pricing optimization |
| **Sports** | Player performance scoring, game outcome prediction |